In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from Utils.load_files import load_binary
from Utils.Sessions import Session
from pathlib import Path

In [2]:
"""
Change all the variables below to match recording.
"""

base_dir = r"/mnt/senzailab/Kai/#Recording/m18"
date: str | int = "260812"
num_of_rec: int = 2

subfolder_filler = f"{date}_{num_of_rec}"
base_dir = f"{base_dir}/{date}/{subfolder_filler}"

# 0 indexed OneBox ADC channel number
camera_input_channel: int = 1
camera_pulse_threshold: int = 11000

"""------------------------------------------------------------------"""
oebin_file_dir = next((Path(base_dir) / str(date)).glob("*/experiment1/recording1/structure.oebin"))

session = Session(oebin_file_dir)
session_info = session.get_session_info()

base_data_dir = session_info['base_path']
record_nodes: str = session_info['record_nodes']
recording_name: str = session_info['recording_name']
experiment_id: str = session_info['experiment_id']

path_between = f'/{record_nodes}/{experiment_id}/'
continuous_folder = base_data_dir + path_between + recording_name + '/continuous/'

print("==============================")
print("Loading camera pulse data...")
ADC_name: str = session_info['continuous_ADC_folder']
analogue_total_input_channel_number: int = session_info['ADC_input_channel']
ADC_datafile = continuous_folder + ADC_name + '/continuous.dat'

camera_data = load_binary(
    ADC_datafile,
    analogue_total_input_channel_number,
    camera_input_channel,
)

ADC_continuous_time_stamp_file = f'{continuous_folder}/{ADC_name}/timestamps.npy'
ADC_continuous_time_stamp_data = np.load(
    ADC_continuous_time_stamp_file,
    mmap_mode='r',
)

assert len(ADC_continuous_time_stamp_data) == len(camera_data), (
    f"Length mismatch: {len(ADC_continuous_time_stamp_data)} vs {len(camera_data)}"
)

Loading session from file: /mnt/senzailab/Kai/#Recording/m18/260812/260812_2/260812/Record Node 102/experiment1/recording1/structure.oebin
Loading camera pulse data...


In [3]:
camera_high = camera_data > camera_pulse_threshold
camera_frame_index = np.flatnonzero(
    camera_high & ~np.r_[False, camera_high[:-1]]
)
camera_frame_times = np.asarray(
    ADC_continuous_time_stamp_data[camera_frame_index],
    dtype=np.float64,
)

assert np.all(np.isfinite(camera_frame_times))
assert np.all(np.diff(camera_frame_times) > 0)

motive_csv_file = Path(base_dir) / f"{date}.csv"
with motive_csv_file.open('r', encoding='utf-8-sig') as stream:
    motive_frame_count = sum(1 for _ in stream) - 8

assert len(camera_frame_times) == motive_frame_count, (
    f"Camera pulse and Motive frame counts differ: "
    f"{len(camera_frame_times)} vs {motive_frame_count}"
)

np.save(
    (npyfile_path := f"{base_dir}/data/camera_frame_times.npy"),
    camera_frame_times,
)
print(f"camera_frame_times saved to {npyfile_path}")

camera_frame_times saved to /mnt/senzailab/Kai/#Recording/m18/260812/260812_2/data/camera_frame_times.npy


In [4]:
print("frame count:", len(camera_frame_times))
print("first 10:", camera_frame_times[:10], "\n")
print("last 10:", camera_frame_times[-10:])
print("median frame interval:", np.median(np.diff(camera_frame_times)))

frame count: 392057
first 10: [9.68692847 9.69527747 9.70359348 9.71194249 9.7202585  9.72860751
 9.73692352 9.74527252 9.75358853 9.76193754] 

last 10: [3276.54404528 3276.55236129 3276.5607103  3276.56902631 3276.57737532
 3276.58569132 3276.59404033 3276.60235634 3276.61070535 3276.61902136]
median frame interval: 0.008349009438916255
